# 复现 Claude 的智能体搜索基准测试得分

Claude 的[已发布的智能体搜索得分](https://www.anthropic.com/system-cards)（DeepSearchQA、BrowseComp）**在公开的 Messages API 上是可以复现的**。关键在于运行框架（harness）的配置：少数几个 API 参数在短对话中无关紧要，可一旦智能体在数十万词元上执行 30 多次工具使用，它们就成为关键支撑。

第三方评估结果偏低的一个常见原因就是运行框架配置。本指南逐一讲解每个参数，并说明它为何重要。

> **成本与运行时间**：本笔记本中的实时单元格运行 **3 个演示问题**（约 5–10 分钟，花费几美元的 API 费用）。复现完整的 900 题基准测试需要几百美元，并在中等并发下运行几个小时；本笔记本不会重新执行完整运行。


## 学完本指南后，你将能够：

- **构建**一个智能体搜索循环，使用程序化工具使用、服务端压缩和任务预算，复现 Claude 已发布的智能体搜索得分。
- **理解**每个配置选择对长周期智能体任务为何重要。
- **改造**同一运行框架以适配 BrowseComp，或你自己的深度研究基准测试，只需替换数据集、几行配置和评分器。


## 前置条件

**所需知识**
- 熟悉 Messages API 和基本工具使用
- 了解智能体循环的概念（调用模型 → 处理工具使用 → 再次调用）

**所需工具**
- 需要 Python 3.11+
- 需要 `anthropic >= 0.111.0`
- 拥有 Anthropic API 密钥，并可访问 `web_search`、`web_fetch` 和 `code_execution` 服务端工具（[如何启用](https://platform.claude.com/docs/en/agents-and-tools/tool-use/code-execution-tool)）

**建议具备**
- [程序化工具使用指南](https://platform.claude.com/cookbook/tool-use-programmatic-tool-calling-ptc)，它介绍了我们在此使用的模式
- [自动上下文压缩指南](https://platform.claude.com/cookbook/tool-use-automatic-context-compaction)，我们大量依赖它


## 环境准备


In [1]:
%%capture
%pip install -U "anthropic>=0.111.0" pandas python-dotenv

In [2]:
import time

import anthropic
import pandas as pd
from dotenv import load_dotenv
from utils.agentic_search import (
    SAMPLE_QUESTIONS,
    USER_PROMPT_TEMPLATE,
    extract_result_tag,
    grade_response,
    summarize_response,
)

load_dotenv()

MODEL = "claude-sonnet-5"
# Hold the grader model fixed so scores are comparable across the model under test.
GRADER_MODEL = "claude-opus-4-6"

# With server-side tools, a single streamed call can run for many minutes of
# in-sandbox tool execution before the first token arrives. We raise the read
# timeout to 60 minutes; with streaming this is a per-event ceiling, not a
# whole-response one, so it only matters for the gap before the first event.
client = anthropic.Anthropic(
    max_retries=20,
    timeout=anthropic.Timeout(5.0, read=3600.0, write=600.0, pool=600.0),
)

## 任务

下面是三道采用 DeepSearchQA 格式的演示问题。每一题都可以作答，但需要经过多轮搜索、阅读和交叉核对：


In [3]:
for q in SAMPLE_QUESTIONS:
    print(f"[{q['example_id']}] ({q['answer_type']})")
    print(f"  Q: {q['problem']}")
    print(f"  A: {q['answer']}\n")

[demo-0] (Single Answer)
  Q: Among the chief executives of the four largest US banks by total assets as of Q1 2024, which one had held their CEO position the longest?
  A: Jamie Dimon

[demo-1] (Set Answer)
  Q: Which three countries had the largest installed offshore wind capacity at the end of 2023?
  A: China, United Kingdom, Germany

[demo-2] (Set Answer)
  Q: List the films that won the Palme d'Or at the Cannes Film Festival in 2021, 2022, and 2023, and name the director of each.
  A: Titane (Julia Ducournau), Triangle of Sadness (Ruben Östlund), Anatomy of a Fall (Justine Triet)



我们将每道问题包装进一段简短的提示词，要求 Claude 自由规划，并把最终答案放入 `<result>` 标签，以便评分器能够清晰地提取：


In [4]:
print(USER_PROMPT_TEMPLATE.format(question=SAMPLE_QUESTIONS[0]["problem"]))

I want you to answer the following question.

<question>Among the chief executives of the four largest US banks by total assets as of Q1 2024, which one had held their CEO position the longest?</question>

First plan out your response. This part can be as long as needed. You may need to run many searches, this is totally fine.

Then provide a short and concise answer in <result> tags. For questions expecting multiple answers, separate them with commas.


模型需要规划、上网搜索、抓取并阅读页面、运行计算，并持续这样做直到有把握，然后给出一个干净、可评分的最终答案。这句话中几乎每一部分都是粗糙的运行框架会悄悄丢分的地方。我们将逐块搭建这个循环。

> 上面的三道问题是为本指南撰写的（并非取自 DeepSearchQA-900），目的是让实时单元格快速且低成本。真正的基准测试见下文*数据集*。

### 数据集

| 基准测试 | 规模 | 获取途径 |
|---|---|---|
| **BrowseComp** | 1,266 个问题 | [Wei 等，2025](https://arxiv.org/html/2504.12516v1)（OpenAI） |
| **DeepSearchQA** | 900 个问题 | [Kaggle Benchmarks](https://www.kaggle.com/benchmarks/google/dsqa)（[论文](https://arxiv.org/abs/2601.20975)） |

运行框架期望每一行包含 `problem`、`answer` 和 `answer_type` 字段；加载数据时请据此映射源数据模式。


## 1. 工具：程序化工具使用

最大的效率杠杆是**[程序化工具使用](https://platform.claude.com/docs/en/agents-and-tools/tool-use/programmatic-tool-calling)**（PTC）：Claude 不再直接调用 `web_search` 和 `web_fetch`、把每个结果通过 API 往返一次，而是在 `code_execution` 沙箱中编写 Python，并*从代码内部*调用搜索/抓取。只有代码打印出的摘要返回到对话中。

对深度研究而言这一点至关重要。单个问题可能发起 50 多次抓取。没有 PTC，每一个页面正文都会进入上下文；有了 PTC，模型在沙箱内阅读它们，只呈现所需内容。

我们的启用方式是：把 `code_execution` 列为唯一可直接调用的工具，并将 `web_search`/`web_fetch` 标注为*可从代码执行调用*，其结果从响应中排除：


In [5]:
TOOLS = [
    {
        "type": "code_execution_20260521",
        "name": "code_execution",
    },
    {
        "type": "web_search_20260318",
        "name": "web_search",
        "max_uses": 10_000,
        "allowed_callers": ["code_execution_20260521"],
        "response_inclusion": "excluded",
    },
    {
        "type": "web_fetch_20260318",
        "name": "web_fetch",
        "max_uses": 10_000,
        "max_content_tokens": 1_000_000,
        "allowed_callers": ["code_execution_20260521"],
        "response_inclusion": "excluded",
    },
]

BETAS = [
    "compact-2026-01-12",
    "task-budgets-2026-03-13",
]

有两个细节值得强调：

- **`code_execution_20260521`** 使用的工具提示词会披露沙箱每个单元格的墙钟时间上限，因此 Claude 会把长时间运行的搜索拆成更短的单元格，而不是写一个会超时的大循环。
- **`response_inclusion: "excluded"`** 把 `web_search` 和 `web_fetch` 的结果留在沙箱内；响应中的其余内容（`thinking`、`text`、`server_tool_use`、代码执行结果、`compaction`）在第 3 节中随 `model_dump()` 一起往返。


## 2. 思考、强度、预算与压缩

四个请求级参数告诉 Claude *该多努力*以及*如何管理自己的上下文*：


In [6]:
THINKING = {"type": "adaptive"}

OUTPUT_CONFIG = {
    "effort": "max",
    "task_budget": {"type": "tokens", "total": 3_000_000},
}

# Match the system-card configuration. To reproduce a different model's score,
# use the effort tier and compaction trigger from that model's model card.
COMPACTION_TRIGGER = 200_000

COMPACT_INSTRUCTIONS = (
    "Your summary MUST begin by restating the user's ORIGINAL QUESTION "
    "verbatim and in full, wrapped in <original_question> and "
    "</original_question> tags. Then summarize the research progress so "
    "far: key sources found, data extracted, partial answers, and what "
    "remains to be done. Do NOT call any tools. Your summary MUST also "
    "include this instruction verbatim: 'Provide your final answer "
    "wrapped in <result> and </result> tags.'"
)

CONTEXT_MANAGEMENT = {
    "edits": [
        {
            "type": "compact_20260112",
            "trigger": {"type": "input_tokens", "value": COMPACTION_TRIGGER},
            "instructions": COMPACT_INSTRUCTIONS,
        }
    ]
}

### 为什么 `COMPACT_INSTRUCTIONS` 如此重要

当对话达到触发阈值时，API 会对其进行压缩：一次独立的模型调用总结历史，智能体从摘要继续执行。默认的摘要提示词在设计上与任务无关；它并不知道*对基准测试而言*，原始问题和答案格式要求是必须保留下来的两样东西。[自定义 `instructions`](https://platform.claude.com/docs/en/build-with-claude/compaction#custom-summarization-instructions) 就是告诉它这一点的方式。

如果没有这些指令，压缩后的智能体只有*它找到了什么*的摘要，却没有*它被问了什么*，在长问题上它常常要求用户复述问题，从而得零分。

这一影响随压缩触发的频率而扩大：在低触发阈值时最明显（单个问题可能压缩多次），在本指南使用的 200K 触发阈值时较小，但无论如何它都是运行框架中最便宜的保险。

> **说明**：最新的 Claude 模型卡报告的 DeepSearchQA *不使用*压缩（该模型有 100 万词元的上下文窗口，因此 100 万词元预算的任务可以放进一个窗口），而 BrowseComp 是在 300 万词元总预算下*使用*压缩报告的。这里我们采用 300 万词元预算的 BrowseComp 配置，使压缩有意义；在 DeepSearchQA 上，多数问题仍远低于 100 万词元，不会触发压缩。

其余三项：

| 参数 | 原因 |
|---|---|
| `thinking: adaptive` | 让 Claude 在结果出人意料时于工具使用之间思考，不出人意料时跳过思考 |
| `effort` | 模型卡上的数字是在模型最高思考强度档位下报告的 |
| `task_budget.total` | 告诉 Claude 它在所有轮次和压缩中的累计输出预算，使其能够自行配速，而不是提前放弃 |


## 3. 往返循环

使用服务端工具时，每次 API 调用内部可以运行多次工具迭代，然后以 `stop_reason="pause_turn"` 返回以存档。我们在客户端的工作只是：取响应内容，将其作为一轮助手消息追加，然后再次调用。我们持续进行直到 `stop_reason="end_turn"`。

有一个辅助函数可以让请求体在各轮次之间保持整洁：


In [7]:
def truncate_to_last_compaction(messages: list[dict]) -> list[dict]:
    """Drop history that precedes the most recent compaction block.

    The server already discarded that history when it compacted; resending it is
    pure overhead and on long runs can exceed the 32 MB request-body limit.
    `messages[0]` (the original user prompt) is always kept so the request still
    starts with a user turn.
    """
    for mi in range(len(messages) - 1, 0, -1):
        content = messages[mi].get("content")
        if not isinstance(content, list):
            continue
        for bi in range(len(content) - 1, -1, -1):
            blk = content[bi]
            if isinstance(blk, dict) and blk.get("type") == "compaction":
                trimmed = {**messages[mi], "content": content[bi:]}
                return [messages[0], trimmed, *messages[mi + 1 :]]
    return messages

现在是循环本身。它是一个单一函数，篇幅较长，但每一行都做着实测运行真正需要的事：


In [8]:
def sample(question: str, *, max_turns: int = 100) -> dict:
    """Run one question to completion through the agentic search loop."""
    # cache_control on the user message turns on prompt caching for the whole
    # request, so the server-side tool loop caches between iterations.
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": USER_PROMPT_TEMPLATE.format(question=question),
                    "cache_control": {"type": "ephemeral"},
                }
            ],
        }
    ]
    request = {
        "model": MODEL,
        "max_tokens": 64_000,
        "tools": TOOLS,
        "betas": BETAS,
        "thinking": THINKING,
        "output_config": OUTPUT_CONFIG,
        "context_management": CONTEXT_MANAGEMENT,
        "messages": messages,
    }

    usage = {"input_tokens": 0, "output_tokens": 0, "cache_read_input_tokens": 0}
    transcript: list[str] = []
    tool_calls = 0
    start = time.time()

    for turn in range(max_turns):
        # Stream so the per-event read-timeout applies, not a per-response one.
        with client.beta.messages.stream(**request) as stream:
            response = stream.get_final_message()

        usage["input_tokens"] += response.usage.input_tokens
        usage["output_tokens"] += response.usage.output_tokens
        usage["cache_read_input_tokens"] += (
            getattr(response.usage, "cache_read_input_tokens", 0) or 0
        )
        tool_calls += sum(
            1 for b in response.content if getattr(b, "type", "") == "server_tool_use"
        )
        transcript.append(f"--- turn {turn} ---\n" + summarize_response(response))

        # Persist the code-execution container so state survives across turns.
        if getattr(response, "container", None) is not None:
            request["container"] = response.container.id

        # Round-trip: append assistant content, drop pre-compaction history.
        messages.append(
            {
                "role": "assistant",
                "content": [b.model_dump(exclude_none=True) for b in response.content],
            }
        )
        messages[:] = truncate_to_last_compaction(messages)

        if response.stop_reason == "end_turn":
            final_text = "".join(
                b.text for b in response.content if getattr(b, "type", "") == "text"
            )
            return {
                "text": final_text,
                "result": extract_result_tag(final_text),
                "turns": turn + 1,
                "tool_calls": tool_calls,
                "elapsed_s": time.time() - start,
                "usage": usage,
                "transcript": transcript,
            }
        if response.stop_reason not in ("pause_turn", "max_tokens"):
            raise RuntimeError(f"unexpected stop_reason={response.stop_reason}")

    raise RuntimeError(f"did not finish within {max_turns} turns")

### 重试

SDK 会在连接错误、`408`、`409`、`429` 和 `≥500` 响应时[自动重试](https://platform.claude.com/docs/en/api/sdks/python#retries)，采用指数退避，并遵守 API 在速率限制响应中发送的 `Retry-After` 头。默认是 2 次重试；对于完整的基准测试运行，我们在客户端上设置 `max_retries=20`（见上文*环境准备*），这样运行中的短暂过载不会把某道题判为零分。

> **识别悄悄发生的工具限流**：服务端工具的速率限制与模型词元限制是分开的。耗尽它们不会抛出 API 错误；`too_many_requests` 表现为响应内部的一个工具结果错误，Claude 通常会在沙箱内重试。如果一次完整的基准测试运行没有异常却得分意外偏低，请检查 `result["transcript"]` 中打印 `rate_limit_error` 或 `too_many_requests` 的代码执行单元格。客户端侧的解决办法是降低并发（或联系 Anthropic 支持以提高工具速率限制）；客户端重试没有帮助，因为 API 调用返回了 200。
>
> 同样，把每个问题的 `sample()` 调用包在 `try/except` 中，这样单次 `stop_reason="refusal"` 或不可恢复的错误只会让该题得零分，而不会中止整个运行。


## 4. 端到端运行一个问题

我们来运行第一个问题。预计这个单元格需要一到两分钟：在那一次 `stream()` 调用背后，Claude 正在多次编写并执行搜索代码。


In [9]:
result = sample(SAMPLE_QUESTIONS[0]["problem"])

print(f"tool_calls={result['tool_calls']}  elapsed={result['elapsed_s']:.0f}s")
print(
    f"tokens: in={result['usage']['input_tokens']:,}  out={result['usage']['output_tokens']:,}  "
    f"cache_read={result['usage']['cache_read_input_tokens']:,}"
)
print()
print("Final <result>:", result["result"])

tool_calls=4  elapsed=56s
tokens: in=130  out=3,842  cache_read=39,703

Final <result>: **Jamie Dimon**, CEO of **JPMorgan Chase**, had held his CEO position the longest among the chief executives of the four largest US banks by total assets (JPMorgan Chase, Bank of America, Citigroup, and Wells Fargo) as of Q1 2024. Dimon became JPMorgan Chase's CEO on January 1, 2006 — roughly 18 years by Q1 2024 — compared to Brian Moynihan (Bank of America CEO since January 2010), Charles Scharf (Wells Fargo CEO since October 2019), and Jane Fraser (Citigroup CEO since March 2021).


转录记录展示了工作的形态。每一轮都是一批沙箱内工具使用之后的 `pause_turn` 检查点：


In [10]:
for line in result["transcript"]:
    print(line + "\n")

--- turn 0 ---
  [thinking]  963 chars
  [text]      I'll research this step by step: first identifying the four largest US banks by total assets as of Q
  [tool_use]  code_execution
  [code_execution_tool_result]
  [thinking]  82 chars
  [tool_use]  code_execution
  [code_execution_tool_result]
  [thinking]  511 chars
  [tool_use]  code_execution
  [code_execution_tool_result]
  [thinking]  1365 chars
  [tool_use]  code_execution
  [code_execution_tool_result]
  [thinking]  700 chars
  [text]      Based on my research, I now have sufficient information to answer this question.  **Summary of findi
  [text]      "JPMorgan Chase & Co., the largest US bank by total assets, reported a sequential increase of $215.3
  [text]       were the other top banks, a ranking that held consistently across 2024.  Their respective CEOs and 
  [text]      Dimon became CEO on January 1, 2006 and one year later also became Chairman of the Board.
  [text]        2. **Bank of America – Brian Moynihan**: 
  [

## 5. 评分

基准测试评分使用模型评判式的 F1 评分器。评分器看到问题、标准答案和抽取出的 `<result>` 文本，然后报告哪些标准答案条目被找到、哪些不在标准答案集中的额外条目出现了。据此我们计算精确率、召回率和 F1。

有两件事必须做对：

- **评分的是 `<result>` 标签，而不是整个响应。** 完整响应包含规划性文字和搜索摘要；把这些交给评分器会同时放大假阳性和假阴性。`extract_result_tag()` 只提取最终答案。
- **固定评分模型。** 无论 `MODEL` 是什么，我们都使用 `GRADER_MODEL = "claude-opus-4-6"`。用被测模型评分会让各模型之间的得分不可比较。

评分提示词很直接：

~~~text 你的任务是判断某个响应是否得出了正确答案。

问题：<question>{question}</question> 正确答案（类型：{answer_type}）：<correct_answer>{answer}</correct_answer> 待评估的响应：<response>{response}</response>

对每个预期答案条目，指出它是否出现在响应中。然后列出响应中不在正确答案清单里的任何答案。措辞不必完全一致。

请按这一精确的 XML 格式回复：<evaluation> <explanation>一句话</explanation> <correctness_details> <item answer="expected_item_1" correct="true|false"/> </correctness_details> <excessive_answers> <item>extra_item_if_any</item> </excessive_answers> </evaluation> ~~~

解析以及精确率/召回率/F1 的计算都在 [`utils/agentic_search.py`](utils/agentic_search.py) 中；这里我们只需调用它：


In [11]:
grade = grade_response(client, GRADER_MODEL, SAMPLE_QUESTIONS[0], result["text"])
print(f"answer    : {grade['extracted_answer']}")
print(f"precision : {grade['precision']:.2f}")
print(f"recall    : {grade['recall']:.2f}")
print(f"F1        : {grade['f1']:.2f}")

answer    : **Jamie Dimon**, CEO of **JPMorgan Chase**, had held his CEO position the longest among the chief executives of the four largest US banks by total assets (JPMorgan Chase, Bank of America, Citigroup, and Wells Fargo) as of Q1 2024. Dimon became JPMorgan Chase's CEO on January 1, 2006 — roughly 18 years by Q1 2024 — compared to Brian Moynihan (Bank of America CEO since January 2010), Charles Scharf (Wells Fargo CEO since October 2019), and Jane Fraser (Citigroup CEO since March 2021).
precision : 1.00
recall    : 1.00
F1        : 1.00


## 6. 运行演示集

现在运行全部三个问题，并用一张对比表展示用量和 F1。（要运行完整的 900 题基准测试，请把 `SAMPLE_QUESTIONS` 替换为完整数据集并加入并发；见*下一步*。）


In [12]:
rows = []
for q in SAMPLE_QUESTIONS:
    r = sample(q["problem"])
    g = grade_response(client, GRADER_MODEL, q, r["text"])
    rows.append(
        {
            "id": q["example_id"],
            "tool_calls": r["tool_calls"],
            "in_tokens": r["usage"]["input_tokens"],
            "out_tokens": r["usage"]["output_tokens"],
            "f1": g["f1"],
            "answer": g["extracted_answer"][:60],
        }
    )

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nMean F1: {df['f1'].mean():.2f}")
# in_tokens is the uncached billed input; with caching on, most of the actual
# context is cache reads (tracked separately in usage.cache_read_input_tokens).
print(f"Total tokens: {df['in_tokens'].sum():,} in / {df['out_tokens'].sum():,} out")

    id  tool_calls  in_tokens  out_tokens  f1                                                        answer
demo-0          11        312       10503 1.0  **Jamie Dimon**, CEO of **JPMorgan Chase** (the largest US b
demo-1           7        208        9768 1.0  The three countries with the largest installed offshore wind
demo-2           3        104        1592 1.0 - **2021**: *Titane* — directed by Julia Ducournau\n- **2022*

Mean F1: 1.00
Total tokens: 624 in / 21,863 out


## 7. 扩展到完整基准测试

用本笔记本的配置跑完完整的 DeepSearchQA-900 和 BrowseComp-1266 数据集，就能复现已公布的模型卡得分。各模型已公布的智能体搜索得分，见各模型的 [Claude 模型卡](https://www.anthropic.com/system-cards)。


## 8. 适配 BrowseComp

BrowseComp 的问题是单答案（“哪部 1995 年的电影……”），而非集合型答案，因此运行框架中唯一需要改动的就是评分器：


In [13]:
from utils.agentic_search import grade_browsecomp

# Grading: a single-letter judge (A=correct, B=incorrect, C=abstain).
# accuracy = fraction of "A" verdicts.
bc_grade = grade_browsecomp(client, GRADER_MODEL, SAMPLE_QUESTIONS[0], result["text"])
print(bc_grade)

{'example_id': 'demo-0', 'extracted_answer': "**Jamie Dimon**, CEO of **JPMorgan Chase**, had held his CEO position the longest among the chief executives of the four largest US banks by total assets (JPMorgan Chase, Bank of America, Citigroup, and Wells Fargo) as of Q1 2024. Dimon became JPMorgan Chase's CEO on January 1, 2006 — roughly 18 years by Q1 2024 — compared to Brian Moynihan (Bank of America CEO since January 2010), Charles Scharf (Wells Fargo CEO since October 2019), and Jane Fraser (Citigroup CEO since March 2021).", 'grader_letter': 'A', 'accuracy': 1.0}


工具、beta 标志和 `sample()` 循环保持不变。如果你想要严格复现，请查阅模型卡中 BrowseComp 公布时所用的确切思考强度档位和压缩触发条件；§2 中的值是不错的默认设置。

关键在于：一旦循环正确，换基准测试就是换一个数据集、几行配置和一个评分器。


## 回顾

你构建了一个智能体搜索循环，它会：

- 从代码执行中**以编程方式**调用 `web_search` 和 `web_fetch`，因此 50 多次页面抓取不会淹没上下文（§1）。
- 告诉 Claude **该下多大力**（`effort`、`task_budget`）以及**该怎么思考**（`adaptive`）（§2）。
- 通过**在压缩提示词中重述问题**来挺过压缩；在压缩频繁发生时，这是最有效的单一杠杆（§2）。
- 正确往返 `pause_turn` 响应，持久化代码容器并裁剪压缩前的历史（§3）。
- 用模型评判式的 F1 评分器评分，并复现已公布的 DeepSearchQA 得分（§5–7）。


## 下一步

- **扩展到完整基准测试**：把 `sample()` 包在 `concurrent.futures.ThreadPoolExecutor` 中（客户端是线程安全的）；SDK 内置的重试已经覆盖 429/529。在并发 50 下，DeepSearchQA-900 大约 2 小时完成。
- **尝试多智能体变体**：[异步多智能体编排指南](https://platform.claude.com/cookbook/patterns-agents-async-multi-agent-orchestration) 展示了 N 个智能体的团队如何以更低时延超过这个单智能体循环的准确率。
- **移植到你自己的基准测试**：约定是 `[{problem, answer, answer_type}]` 这样的行、几行配置和一个评分函数。循环不变。
- **阅读模型卡**：各模型的 [Claude 模型卡](https://www.anthropic.com/system-cards) 记录了每个已公布的基准测试数字所用的配置。
